In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras import (
    layers,
    models,
    datasets,
    callbacks,
    losses,
    optimizers,
    metrics,
)

# from tensorflow.python.framework.ops import disable_eager_execution
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense, Conv2DTranspose, Cropping2D, UpSampling2D, Reshape
from tensorflow.keras.applications import ResNet50, InceptionResNetV2, MobileNetV2

# disable_eager_execution()

In [ ]:
#def sample_batch(dataset):
#    batch = dataset.take(1).get_single_element()
#    if isinstance(batch, tuple):
#        batch = batch[0]
#    return batch.numpy()


#def display(
#    images, n=10, size=(20, 3), cmap="gray_r", as_type="float32", save_to=None):
#    """
#    Displays n random images from each one of the supplied arrays.
 #   """
  #  if images.max() > 1.0:
   #     images = images / 255.0
    #elif images.min() < 0.0:
     #   images = (images + 1.0) / 2.0

   # plt.figure(figsize=size)
    #for i in range(n):
     #   _ = plt.subplot(1, n, i + 1)
      #  plt.imshow(images[i].astype(as_type), cmap=cmap)
       # plt.axis("off")

    #if save_to:
     #   plt.savefig(save_to)
      #  print(f"\nSaved to {save_to}")

   # plt.show()

# Designing and Building Variational Autoencoder

creating  a new Sampling layer for sampling from the distribution defined by z_mean and z_log_var.

In [ ]:
#Sampling Layer

class Sampling(layers.Layer):
    #We create a new layer by subclassing the keras base Layer
    def call(self, inputs):
        #
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = K.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

In [ ]:
# Encoder

input_shape = (400, 320, 3)  

# Load ResNet50 as the base model
#base_model = InceptionResNetV2(include_top=False, weights='imagenet', input_shape=input_shape)
#base_model = InceptionResNetV2(include_top=False, weights='imagenet', input_shape=input_shape)

# Freeze the base_model
#base_model.trainable = False

embed_dim = 512
encoder_input = layers.Input(shape=input_shape, name="encoder_input")

# Load InceptionResNetV2 model pretrained on ImageNet without the top layer
base_model = InceptionResNetV2(include_top=False, weights='imagenet', input_tensor=encoder_input)

# Since we are using it in a VAE, we do not need to train the InceptionResNetV2 layers
base_model.trainable = False

# Adding the final part of the encoder
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)  # This replaces the flattening and reduces the dimensionality

# Instead of connecting the Flatten layer directly to the 2D latent space,
# we connect it to layers z_mean and z_log_var.
z_mean = layers.Dense(embed_dim, name="z_mean")(x)
z_log_var = layers.Dense(embed_dim, name="z_log_var")(x)




#The Sampling layer samples a point z in the latent space from the
#normal distribution defined by the parameters z_mean and z_log_var.
z = Sampling()([z_mean, z_log_var])

#The Keras Model that defines the encoder—a model that takes an input
#image and outputs z_mean, z_log_var and a sampled point z from the
#normal distribution defined by these parameters.
encoder = models.Model(encoder_input, [z_mean, z_log_var, z], name="encoder")
encoder.summary()

In [ ]:
target_shape = (8, 10, 64)
# Decoder
decoder_input = layers.Input(shape=(embed_dim,), name="decoder_input")

x = Dense(target_shape[0] * target_shape[1] * target_shape[2], activation='relu')(decoder_input)
x = Reshape(target_shape)(x)

x = Conv2DTranspose(256, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
x = Conv2DTranspose(128, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
x = Conv2DTranspose(64, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
x = Conv2DTranspose(64, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
x = Conv2DTranspose(32, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
x = Conv2DTranspose(16, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
x = Conv2D(3,(3, 3),activation="sigmoid",padding="same")(x)
decoder_output = Cropping2D(cropping=((56,56), (160,160)),name="decoder_output")(x)

decoder = models.Model(decoder_input, decoder_output)
decoder.summary()























# To Train Variational Autoencoder

In [ ]:
#To create the VAE model, we extend the Keras Model class, enabling us to incorporate the KL
#divergence calculation
#into a custom train_step method.

In [ ]:
class VAE(models.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = metrics.Mean(
            name="reconstruction_loss"
        )
        self.kl_loss_tracker = metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def call(self, inputs):
        """Call the model on a particular input."""
        z_mean, z_log_var, z = encoder(inputs)
        reconstruction = decoder(z)
        return z_mean, z_log_var, reconstruction

    def train_step(self, data):
        """Step run during training."""
        with tf.GradientTape() as tape:
            #TensorFlow's Gradient Tape helps calculate gradients during a forward pass.
            #To use it, we need wrap the code that performs the operations you want to differentiate within a tf.GradientTape() context.
            #After recording the operations, we can compute the gradient of the loss function concerning certain variables using tape.gradient().
            #These gradients are then used to update the variables with the optimizer.

            z_mean, z_log_var, reconstruction = self(data)
            beta = 500
            reconstruction_loss = tf.reduce_mean(
                beta
                * losses.mean_squared_error(
                    data, reconstruction)
            )
            kl_loss = tf.reduce_mean(
                tf.reduce_sum(
                    -0.5
                    * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)),
                    axis=1,
                )
            )
            total_loss = reconstruction_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        
        

        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        """Step run during validation."""
        if isinstance(data, tuple):
            data = data[0]

        z_mean, z_log_var, reconstruction = self(data)
        beta = 500
        reconstruction_loss = tf.reduce_mean(
            beta
            * losses.mean_squared_error(data, reconstruction)
        )
        kl_loss = tf.reduce_mean(
            tf.reduce_sum(
                -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)),
                axis=1,
            )
        )
        total_loss = reconstruction_loss + kl_loss

        return {
            "loss": total_loss,
            "reconstruction_loss": reconstruction_loss,
            "kl_loss": kl_loss,
        }

In [ ]:
vae = VAE(encoder, decoder)

#  Training 

In [ ]:
# Compile the variational autoencoder
optimizer = optimizers.Adam(learning_rate=0.0005)
vae.compile(optimizer=optimizer)

In [ ]:
# Create a model save checkpoint
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath="./checkpoint",
    save_weights_only=False,
    save_freq="epoch",
    monitor="loss",
    mode="min",
    save_best_only=True,
    verbose=0,
)
tensorboard_callback = callbacks.TensorBoard(log_dir="./logs")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(rescale=1./255)
    
path_train = "/path/to/private-data/Datagen_Left/Outlet_Left/train"
path_val = "/path/to/private-data/Datagen_Left/Outlet_Left/validation"
    
batch_size = 32
target_size = (input_shape[0],input_shape[1])
train_generator = datagen.flow_from_directory(path_train, target_size=target_size,  batch_size=batch_size, class_mode=None, shuffle=True, color_mode='rgb')

validation_generator = datagen.flow_from_directory(path_val, target_size=target_size , batch_size=batch_size, class_mode=None, shuffle=True, color_mode='rgb')

In [ ]:
epochs = 100
batch_size = 32

vae.fit(
    train_generator,
    epochs=epochs,
    batch_size=batch_size,
    shuffle=True,
    validation_data=validation_generator,
    callbacks=[model_checkpoint_callback, tensorboard_callback],
)

In [ ]:
#history = vae.history

In [ ]:
#print(history.history)

In [ ]:
#plt.plot(history['loss'], label='Train Loss')
#plt.plot(history['val_loss'], label='Validation Loss')
#plt.title('Training and Validation Loss')
#plt.xlabel('Epoch')
#plt.ylabel('Loss')
#plt.legend()
#plt.show()

## Saving Trained Checkpoints

In [ ]:
# Save the final models
#vae.save("./models/VAE_Outlet_left_inceptionResnet_full")
#encoder.save("./models/VAE_Outlet_left_inceptionResnet_encoder")
#decoder.save("./models/VAE_Outlet_left_inceptionResnet_decoder")




In [ ]:
from tensorflow.keras.models import load_model

vae = load_model("./models/outlet")
encoder=load_model("./models/outlet_encoder")
decoder=load_model("./models/outlet_decoder")



In [ ]:
#train_generator[1].shape
encoder.summary()

In [ ]:
def compute_elbo_losses(original_images, reconstructed_images, z_mean, z_logvar):

    # Compute the Mean Squared Error (MSE) loss for each image
    mse_loss = np.mean((original_images - reconstructed_images) ** 2, axis=(1, 2, 3))
    
    # Compute the Kullback-Leibler (KL) divergence for each image
    kl_divergence = -0.5 * np.sum(1 + z_logvar - np.square(z_mean) - np.exp(z_logvar), axis=1)
    
    # Compute the ELBO loss for each image
    elbo_loss = mse_loss + kl_divergence
    
    return elbo_loss

# Lets try to reconstruct using our trained variational autoencoder

In [ ]:
#############       From here
import glob, os, cv2
from tensorflow.keras.preprocessing.image import load_img, img_to_array

in3 = []
in5 = []
outlet = []


path = "/path/to/private-data/Croped_Pictures/train/Left/*"
for directory_path in glob.glob(path):
    label=directory_path.split("\\")[-1]
    for image_path in glob.glob(os.path.join(directory_path,"*.bmp")):
        img = load_img(image_path)
        img = img_to_array(img)
        img = img/255.0
        
        if label == "Inlet (3 marking)":
            in3.append(img)

            
        elif label == "Inlet (5 marking)":
            in5.append(img)

            
        elif label == "Outlet":
            outlet.append(img)
print('Done')
in3 = np.array(in3)
in5 = np.array(in5)
outlet =  np.array(outlet)

In [ ]:
val_in3 = []
val_in5 = []
val_outlet = []
val_others= []
 

path = "/path/to/private-data/Croped_Pictures/validation/Left/*"
for directory_path in glob.glob(path):
    label=directory_path.split("\\")[-1]
    for image_path in glob.glob(os.path.join(directory_path,"*.bmp")):
        img = load_img(image_path)
        img = img_to_array(img)
        img = img/255.0
        
        if label == "Inlet (3 marking)":
            val_in3.append(img)

        elif label == "Inlet (5 marking)":
            val_in5.append(img)
            
        elif label == "Outlet":
             val_outlet.append(img)
            
        elif label == "Others":
            val_others.append(img)
print('Done')   
val_in3 = np.array(val_in3)
val_in5 = np.array(val_in5)
val_outlet = np.array(val_outlet)
val_others= np.array(val_others)

In [ ]:
# Create autoencoder predictions and display
# Create autoencoder predictions and display
in3_z_mean, in3_z_log_var, in3_rec = vae.predict(in3)
in5_z_mean, in5_z_log_var, in5_rec = vae.predict(in5)
outlet_z_mean, outlet_z_log_var, outlet_rec = vae.predict(outlet)


# Create autoencoder predictions and display
val_in3_z_mean, val_in3_z_log_var, val_in3_rec = vae.predict(val_in3)
val_in5_z_mean, val_in5_z_log_var, val_in5_rec = vae.predict(val_in5)
val_outlet_z_mean, val_outlet_z_log_var, val_outlet_rec = vae.predict(val_outlet)
val_others_z_mean, val_others_z_log_var, val_others_rec = vae.predict(val_others)







print("Reconstructions")
# display(reconstructions)







print("Reconstructions")
# display(reconstructions)

In [ ]:
outlet_elbo_losses = compute_elbo_losses(outlet, outlet_rec, outlet_z_mean, outlet_z_log_var)
in5_elbo_losses = compute_elbo_losses(in5, in5_rec, in5_z_mean, in5_z_log_var)
in3_elbo_losses = compute_elbo_losses(in3, in3_rec, in3_z_mean, in3_z_log_var)
val_others_elbo_losses = compute_elbo_losses(val_others, val_others_rec, val_others_z_mean, val_others_z_log_var)
val_outlet_elbo_losses = compute_elbo_losses(val_outlet, val_outlet_rec, val_outlet_z_mean, val_outlet_z_log_var)


val_in3_elbo_losses = compute_elbo_losses(val_in3, val_in3_rec, val_in3_z_mean, val_in3_z_log_var)
val_in5_elbo_losses = compute_elbo_losses(val_in5, val_in5_rec, val_in5_z_mean, val_in5_z_log_var)


In [ ]:
plt.imshow(val_outlet[25])

In [ ]:
plt.imshow(val_outlet_rec[25])


In [ ]:
# Function to calculate MSE
def calculate_mse(original, reconstructed):
    return np.mean((original - reconstructed) ** 2)

# Array to store MSE for each image
in5_mse_scores = np.zeros(in5.shape[0])
in3_mse_scores = np.zeros(in3.shape[0])
val_outlet_mse_scores = np.zeros(val_outlet.shape[0])
outlet_mse_scores = np.zeros(outlet.shape[0])
val_others_mse_scores = np.zeros(val_others.shape[0])

# Calculate MSE for each image
for i in range(in5.shape[0]):
   in5_mse_scores[i] = calculate_mse(in5[i], in5_rec[i])
   
for i in range(in3.shape[0]):
   in3_mse_scores[i] = calculate_mse(in3[i], in3_rec[i])

for i in range(outlet.shape[0]):
   outlet_mse_scores[i] = calculate_mse(outlet[i], outlet_rec[i])

for i in range(val_outlet.shape[0]):
   val_outlet_mse_scores[i] = calculate_mse(val_outlet[i], val_outlet_rec[i])

for i in range(val_others.shape[0]):
   val_others_mse_scores[i] = calculate_mse(val_others[i], val_others_rec[i])


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Function to calculate cosine similarity
def calculate_cosine_similarity(original, reconstructed):
    # Flatten the image matrices into vectors
    original_flat = original.flatten().reshape(1, -1)
    reconstructed_flat = reconstructed.flatten().reshape(1, -1)
    # Compute cosine similarity (returns a matrix, get the single value)
    return cosine_similarity(original_flat, reconstructed_flat)[0, 0]

# Assuming the arrays 'in5', 'in3', etc., and their reconstructed counterparts
cosine_scores_in5 = np.zeros(in5.shape[0])
cosine_scores_in3 = np.zeros(in3.shape[0])
cosine_scores_outlet = np.zeros(outlet.shape[0])
cosine_scores_val_outlet = np.zeros(val_outlet.shape[0])
cosine_scores_val_others = np.zeros(val_others.shape[0])

# Calculate cosine similarity for each image
for i in range(in5.shape[0]):
   cosine_scores_in5[i] = calculate_cosine_similarity(in5[i], in5_rec[i])

for i in range(in3.shape[0]):
   cosine_scores_in3[i] = calculate_cosine_similarity(in3[i], in3_rec[i])

for i in range(outlet.shape[0]):
   cosine_scores_outlet[i] = calculate_cosine_similarity(outlet[i], outlet_rec[i])

for i in range(val_outlet.shape[0]):
   cosine_scores_val_outlet[i] = calculate_cosine_similarity(val_outlet[i], val_outlet_rec[i])

for i in range(val_others.shape[0]):
   cosine_scores_val_others[i] = calculate_cosine_similarity(val_others[i], val_others_rec[i])

#print(cosine_scores_in3)  # Output cosine similarities for verification


In [ ]:
import numpy as np
from skimage.metrics import structural_similarity as ssim




#def calculate_ssim(original, reconstructed):
   # Convert images to float32 if necessary
 #   original = tf.cast(original, tf.float32)
  #  reconstructed = tf.cast(reconstructed, tf.float32)
    
   # # Calculate SSIM
    #ssim_value = tf.image.ssim(original, reconstructed, max_val=1.0)
    
    #return ssim_value


def calculate_ssim(original, reconstructed):
    # Convert the images to grayscale if they are RGB
    if original.ndim == 3:
        original = np.mean(original, axis=2)
    if reconstructed.ndim == 3:
        reconstructed = np.mean(reconstructed, axis=2)

    # Calculate SSIM
    return ssim(original, reconstructed, data_range=reconstructed.max() - reconstructed.min())







# Assuming the arrays 'in5', 'in3', etc., and their reconstructed counterparts
ssim_scores_in5 = np.zeros(in5.shape[0])
ssim_scores_in3 = np.zeros(in3.shape[0])
ssim_scores_outlet = np.zeros(outlet.shape[0])
ssim_scores_val_outlet = np.zeros(val_outlet.shape[0])
ssim_scores_val_others = np.zeros(val_others.shape[0])

# Calculate SSIM for each image
for i in range(in5.shape[0]):
    ssim_scores_in5[i] = calculate_ssim(in5[i], in5_rec[i])

for i in range(in3.shape[0]):
    ssim_scores_in3[i] = calculate_ssim(in3[i], in3_rec[i])

for i in range(outlet.shape[0]):
    ssim_scores_outlet[i] = calculate_ssim(outlet[i], outlet_rec[i])

for i in range(val_outlet.shape[0]):
    ssim_scores_val_outlet[i] = calculate_ssim(val_outlet[i], val_outlet_rec[i])

for i in range(val_others.shape[0]):
    ssim_scores_val_others[i] = calculate_ssim(val_others[i], val_others_rec[i])

#print(ssim_scores_in3)  # Output SSIM scores for verification

In [ ]:
import seaborn as sns
plt.figure(figsize=(8, 6))

# Create the box plot
sns.boxplot(data=[in3_elbo_losses, in5_elbo_losses, outlet_elbo_losses, val_outlet_elbo_losses, val_others_elbo_losses])

# Setting labels
#plt.xlabel('Class')
#plt.ylabel('MSE Scores')
plt.title('Elbo Scores with Variational Autoencoders')
plt.xticks([0, 1, 2, 3, 4], ['Inlet type-1', 'Inlet Type-2','Outlet','Outlet (Test)',  'Missplaced & Missing'])  # Label x-ticks with class names

# Show the plot
plt.show()

In [ ]:
import seaborn as sns
plt.figure(figsize=(8, 6))

# Create the box plot
sns.boxplot(data=[in3_mse_scores, in5_mse_scores, outlet_mse_scores, val_outlet_mse_scores, val_others_mse_scores])

# Setting labels
#plt.xlabel('Class')
#plt.ylabel('MSE Scores')
plt.title('MSE Scores with Variational Autoencoders')
plt.xticks([0, 1, 2, 3, 4], ['Inlet type-1', 'Inlet Type-2','Outlet','Outlet (Test)',  'Missplaced & Missing'])  # Label x-ticks with class names

# Show the plot
plt.show()

In [ ]:
import seaborn as sns
plt.figure(figsize=(8, 6))

# Create the box plot
sns.boxplot(data=[cosine_scores_in3, cosine_scores_in5,   cosine_scores_outlet, cosine_scores_val_outlet, cosine_scores_val_others])

# Setting labels
#plt.xlabel('Class')
#plt.ylabel('cosine_similarity Scores')
plt.title('cosine_similarity Scores with Variational Autoencoders')
plt.xticks([0, 1, 2, 3, 4], ['Inlet type-1', 'Inlet Type-2', 'Outlet', 'Outlet (Test)',  'Missplaced & Missing'])  # Label x-ticks with class names

# Show the plot
plt.show()

In [ ]:
import seaborn as sns
plt.figure(figsize=(8, 6))

# Create the box plot
sns.boxplot(data=[ssim_scores_in3, ssim_scores_in5, ssim_scores_outlet, ssim_scores_val_outlet, ssim_scores_val_others])

# Setting labels
#plt.xlabel('Class')
#plt.ylabel('cosine_similarity Scores')
plt.title('SSIM Scores with Variational Autoencoders')
plt.xticks([0, 1, 2, 3, 4], ['Inlet type-1', 'Inlet Type-2', 'Outlet', 'Outlet (Test)',  'Missplaced & Missing'])  # Label x-ticks with class names

# Show the plot
plt.show()

# visualising the Latent Space

In [ ]:
#z_mean, z_var, z = encoder.predict(example_images)
#p = norm.cdf(z)

In [ ]:

# Run this if the below is commented


in5_z_mean, in5_z_log_var,  z_in5 = encoder.predict(in5)
#p_in5 = norm.cdf(z_in5)

in3_z_mean, in3_z_log_var,   z_in3 = encoder.predict(in3)
#p_in3 = norm.cdf(z_in3)

outlet_z_mean, outlet_z_log_var, z_outlet = encoder.predict(outlet)
#p_outlet = norm.cdf(z_outlet)


val_in5_z_mean, val_in5_z_log_var,  val_z_in5 = encoder.predict(val_in5)
#val_p_in5 = norm.cdf(val_z_in5)

val_in3_z_mean, val_in3_z_log_var,  val_z_in3 = encoder.predict(val_in3)
#val_p_in3 = norm.cdf(val_z_in3)

val_outlet_z_mean, val_outlet_z_log_var,  val_z_outlet = encoder.predict(val_outlet)
#val_p_outlet = norm.cdf(val_z_outlet)


val_others_z_mean, val_others_z_log_var,  val_z_others = encoder.predict(val_others)
#val_p_others = norm.cdf(val_z_others)


#-----------------------------------------------------#











In [ ]:
###### run this if only the above is comented
#epsilon = norm.rvs(size=in5_z_mean.shape)
z_in5 = in5_z_mean + np.exp(0.5 * in5_z_log_var) #* epsilon
val_z_in5 = val_in5_z_mean + np.exp(0.5 * val_in5_z_log_var)

#epsilon = norm.rvs(size=in3_z_mean.shape)
z_in3 = in3_z_mean + np.exp(0.5 * in3_z_log_var) #* epsilon
val_z_in3 = val_in3_z_mean + np.exp(0.5 * val_in3_z_log_var) #* epsilon


#epsilon = norm.rvs(size=outlet_z_mean.shape)
z_outlet = outlet_z_mean + np.exp(0.5 * outlet_z_log_var) #* epsilon

#epsilon = norm.rvs(size=val_outlet_z_mean.shape)
val_z_outlet = val_outlet_z_mean + np.exp(0.5 * val_outlet_z_log_var) #* epsilon



#epsilon = norm.rvs(size=val_others_z_mean.shape)
val_z_others = val_others_z_mean + np.exp(0.5 * val_others_z_log_var) #* epsilon


In [ ]:
p_in3 = norm.cdf(z_in3)
p_in5 = norm.cdf(z_in5)
p_outlet = norm.cdf(z_outlet)
val_p_outlet = norm.cdf(val_z_outlet)
val_p_others = norm.cdf(val_z_others)

In [ ]:
in5_label = np.zeros(len(in5))
in3_label = np.ones(len(in3))
outlet_label = np.ones(len(outlet))*2

val_outlet_label = np.ones(len(val_outlet))*3

val_others_label = np.ones(len(val_others))*4



 
label_data=np.concatenate([in5_label,in3_label,outlet_label,val_outlet_label,val_others_label])
#data = np.vstack([in5_z_mean,in3_z_mean,outlet_z_mean,val_outlet_z_mean,val_others_z_mean])   # z mean vecotrs
#data =  np.vstack([in5_z_log_var,in3_z_log_var,outlet_z_log_var,val_outlet_z_log_var,val_others_z_log_var]) # z log distribution vectors
data =  np.vstack([z_in5,z_in3,z_outlet,val_z_outlet,val_z_others])  # z sampled

print(label_data.shape)
print(data.shape)

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import seaborn as sns
import pandas as pd
pca_50 = PCA(n_components=100)
data_50 = pca_50.fit_transform(data)  # Ensure 'data' is defined in your scope

# Apply t-SNE
tsne = TSNE(n_components=2, verbose=1, perplexity=20, n_iter=1000)
tsne_results = tsne.fit_transform(data)

# Create a DataFrame for easier plotting
df_tsne = pd.DataFrame(tsne_results, columns=['tsne-2d-one', 'tsne-2d-two'])
df_tsne['label'] = label_data  # Adding your label data to the DataFrame


# Mapping of numeric labels to descriptive labels
label_map = {
    0: "Inlet Type - 2",
    1: "Inlet Type 1",
    2: "Outlet",
    3: "Outlet (Test)",
    4: "Missplaced & Missing"
}

# Apply the mapping to the label column
df_tsne['label'] = df_tsne['label'].map(label_map)

color_palette = {
    "Inlet Type - 2": "#1f77b4",
    "Inlet Type 1": "#ff7f0e",
    "Outlet": "#2ca02c",
    "Outlet (Test)": "#d62728",
    "Missplaced & Missing": "#9467bd"
}

# Plot using Seaborn for better aesthetics and automatic legend handling
plt.figure(figsize=(10, 8))
scatter = sns.scatterplot(
    x='tsne-2d-one', y='tsne-2d-two',
    hue='label',  # This column is used to map colors to categories
    palette=color_palette,  # Ensures a broad, visually distinct color spectrum
    data=df_tsne,
    legend='full',  # Ensures the legend is displayed
    alpha=0.8,
    s=50  # Adjusts the size of the points in the scatter plot
)

# Enhance the plot with a title and possibly grid lines for better visual parsing
plt.title('Latent z with InceptionRensnet encder in VAE (t-SNE Reduction to 2D)')
plt.grid(True)  # Optional: Adds a grid for easier visualization
plt.show()

In [ ]:
import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import pandas as pd

# Assuming 'data' is your high-dimensional dataset
pca_50 = PCA(n_components=25)
data_50 = pca_50.fit_transform(data)

# Apply t-SNE for 3 dimensions
tsne = TSNE(n_components=3, verbose=1, perplexity=40, n_iter=1000)
tsne_results = tsne.fit_transform(data_50)

# Create a DataFrame for easier plotting
df_tsne = pd.DataFrame(tsne_results, columns=['Dimension 1', 'Dimension 2', 'Dimension 3'])
df_tsne['Label'] = label_data  # your label data

# Map numeric labels to descriptive labels
label_map = {
    0: "Inlet Type-2",
    1: "Inlet Type-1",
    2: "Outlet",
    3: "Outlet (Test)",
    4: "Missplaced & Missing"
}
df_tsne['Label'] = df_tsne['Label'].map(label_map)

# Plot using Plotly Express
fig = px.scatter_3d(
    df_tsne,
    x='Dimension 1',
    y='Dimension 2',
    z='Dimension 3',
    color='Label',  # Color by the 'Label' column
    color_discrete_map={  # Optional: custom color map
        "Inlet Type-2": 'blue',
        "Inlet Type-1": 'orange',
        "Outlet": 'green',
        "Outlet (Test)": 'red',
        "Missplaced & Missing": 'purple'
    },
    title='3D t-SNE Visualization'
)

fig.update_traces(marker=dict(size=3))  # Adjust marker size here
fig.show()


In [ ]:
#------------------------------ run one of this if only not using the z vectors to define anomalies


#z_outlet = outlet_z_log_var
#z_in5 = in5_z_log_var
#z_in3 = in3_z_log_var
#val_z_others = val_others_z_log_var
#val_z_outlet = val_outlet_z_log_var


#z_outlet = outlet_z_mean
#z_in5 = in5_z_mean
#z_in3 = in3_z_mean
#val_z_others = val_others_z_mean
#val_z_outlet = val_outlet_z_mean




centroid = np.mean(z_outlet, axis=0)

In [ ]:
in3_distances = [np.linalg.norm(z - centroid) for z in z_in3]

in5_distances = [np.linalg.norm(z - centroid) for z in z_in5]

outlet_distances = [np.linalg.norm(z - centroid) for z in z_outlet]

val_outlet_distances = [np.linalg.norm(z - centroid) for z in val_z_outlet]

val_others_distances = [np.linalg.norm(z - centroid) for z in val_z_others]





In [ ]:
import seaborn as sns
plt.figure(figsize=(8, 6))

# Create the box plot
sns.boxplot(data=[in3_distances, in5_distances, outlet_distances, val_outlet_distances, val_others_distances])

# Setting labels
#plt.xlabel('Class')
#plt.ylabel('cosine_similarity Scores')
plt.title('Euclidean distance to the mean vector of normal data')
plt.xticks([0, 1, 2, 3, 4], ['Inlet type-1', 'Inlet Type-2', 'Outlet', 'Outlet (Test)',  'Missplaced & Missing'])  # Label x-ticks with class names

# Show the plot
plt.show()

In [ ]:
import numpy as np

# Compute the mean vector
mean_vector = np.mean(z_outlet, axis=0)

# Compute the covariance matrix
covariance_matrix = np.cov(z_outlet, rowvar=False)

epsilon = 1e-10
regularized_cov_matrix = covariance_matrix + epsilon * np.eye(covariance_matrix.shape[0])

from scipy.spatial.distance import mahalanobis
from scipy.linalg import inv

# Inverse of the covariance matrix
inv_cov_matrix = inv(regularized_cov_matrix)

# Function to calculate Mahalanobis distance
def compute_mahalanobis_distance(data_point, mean_vector, inv_cov_matrix):
    diff = data_point - mean_vector
    distance = np.sqrt(diff.T @ inv_cov_matrix @ diff)
    return distance

# Compute Mahalanobis distances for all data points
outlet_mahalanobis_distances = np.array([compute_mahalanobis_distance(point, mean_vector, inv_cov_matrix) for point in z_outlet])
in5_mahalanobis_distances = np.array([compute_mahalanobis_distance(point, mean_vector, inv_cov_matrix) for point in z_in5])
in3_mahalanobis_distances = np.array([compute_mahalanobis_distance(point, mean_vector, inv_cov_matrix) for point in z_in3])
val_outlet_mahalanobis_distances = np.array([compute_mahalanobis_distance(point, mean_vector, inv_cov_matrix) for point in val_z_outlet])
val_others_mahalanobis_distances = np.array([compute_mahalanobis_distance(point, mean_vector, inv_cov_matrix) for point in val_z_others])

val_in3_mahalanobis_distances = np.array([compute_mahalanobis_distance(point, mean_vector, inv_cov_matrix) for point in val_z_in3])
val_in5_mahalanobis_distances = np.array([compute_mahalanobis_distance(point, mean_vector, inv_cov_matrix) for point in val_z_in5])










In [ ]:
import seaborn as sns
plt.figure(figsize=(8, 6))

# Create the box plot
sns.boxplot(data=[in3_mahalanobis_distances, in5_mahalanobis_distances, outlet_mahalanobis_distances, val_outlet_mahalanobis_distances, val_others_mahalanobis_distances])

# Setting labels
#plt.xlabel('Class')
#plt.ylabel('cosine_similarity Scores')
plt.title('Mahalanobis distance between mean vector of normal data')
plt.xticks([0, 1, 2, 3, 4], ['Inlet type-1', 'Inlet Type-2', 'Outlet', 'Outlet (Test)',  'Missplaced & Missing'])  # Label x-ticks with class names

# Show the plot
plt.show()

In [ ]:
from sklearn.mixture import GaussianMixture
gmm = GaussianMixture(n_components=25, covariance_type='full').fit(z_outlet)

gmm_in5 = gmm.score_samples(z_in5)
gmm_in3 =gmm.score_samples(z_in3)
gmm_outlet =gmm.score_samples(z_outlet)
val_gmm_in3 =gmm.score_samples(val_z_in3)
val_gmmin5 =gmm.score_samples(val_z_in5)
val_gmm_outlet =gmm.score_samples(val_z_outlet)
val_gmm_others =gmm.score_samples(val_z_others)

plt.figure(figsize=(8, 6))
sns.boxplot(data=[gmm_in3,gmm_in5,gmm_outlet,val_gmm_outlet,val_gmm_others])

#Setting labels
plt.xlabel('Class')
plt.title('Gaussian Mixture scores')
plt.xticks([0, 1, 2, 3, 4], ['Inlet type-1', 'Inlet Type-2', 'Outlet','Outlet (Test)',   'Misplaced & Missing'])  # Label x-ticks with class names

# Show the plot
plt.show()


In [ ]:
import numpy as np
from sklearn.neighbors import KernelDensity
# Fit a Kernel Density model
kde = KernelDensity(kernel='gaussian', bandwidth=1).fit(z_outlet)

kd_in3= kde.score_samples(z_in3)
kd_in5 = kde.score_samples(z_in5)
Kd_outlet =kde.score_samples(z_outlet)
val_kd_in5=kde.score_samples(val_z_in5)
val_kd_in3=kde.score_samples(val_z_in3)
val_kd_outlet = kde.score_samples(val_z_outlet)
val_kd_others = kde.score_samples(val_z_others)

plt.figure(figsize=(8, 6))
sns.boxplot(data=[kd_in3, kd_in5, Kd_outlet,val_kd_outlet, val_kd_others])

# Setting labels
#plt.xlabel('Class')
#plt.ylabel('cosine_similarity Scores')
plt.title('Kernal Density scores')
plt.xticks([0, 1, 2, 3, 4], ['Inlet type-1', 'Inlet Type-2', 'Outlet','Outlet (Test)',   'Missplaced & Missing'])  # Label x-ticks with class names

# Show the plot
plt.show()



In [ ]:
import numpy as np
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor



# One-Class SVM
ocsvm = OneClassSVM(nu=0.1, kernel="rbf", gamma="auto")
ocsvm.fit(z_outlet)

#ocsvm = OneClassSVM(nu=0.1, kernel="rbf", gamma="auto")
#ocsvm.fit(outlet_z_mean)




In [ ]:
# Isolation Forest
isof = IsolationForest(contamination=0.1, n_estimators=100,max_samples=100)
isof.fit(z_outlet)

#isof = IsolationForest(contamination=0.005, n_estimators=100,max_samples=100)
#isof.fit(outlet_z_mean)




In [ ]:
# Local Outlier Factor
#lof = LocalOutlierFactor(n_neighbors=20, contamination=0.1)
#lof.fit(outlet_z_log_var)


In [ ]:
# One-Class SVM anomaly scores

in5_svm_scores = ocsvm.score_samples(z_in5)
in3_svm_scores = ocsvm.score_samples(z_in3)
outlet_svm_scores = ocsvm.score_samples(z_outlet)
val_outlet_svm_scores = ocsvm.score_samples(val_z_outlet)
val_others_svm_scores = ocsvm.score_samples(val_z_others)


val_in5_svm_scores = ocsvm.score_samples(val_z_in5)
val_in3_svm_scores = ocsvm.score_samples(val_z_in3)



#in5_svm_scores = ocsvm.score_samples(in5_z_mean)
#in3_svm_scores = ocsvm.score_samples(in3_z_mean)
#outlet_svm_scores = ocsvm.score_samples(outlet_z_mean)
#val_outlet_svm_scores = ocsvm.score_samples(val_outlet_z_mean)
#val_others_svm_scores = ocsvm.score_samples(val_others_z_mean)




# Isolation Forest anomaly scores

in5_isof_scores = isof.score_samples(z_in5)
in3_isof_scores = isof.score_samples(z_in3)
outlet_isof_scores = isof.score_samples(z_outlet)
val_outlet_isof_scores = isof.score_samples(val_z_outlet)
val_others_isof_scores = isof.score_samples(val_z_others)



#in5_isof_scores = isof.score_samples(in5_z_mean)
#in3_isof_scores = isof.score_samples(in3_z_mean)
#outlet_isof_scores = isof.score_samples(outlet_z_mean)
#val_outlet_isof_scores = isof.score_samples(val_outlet_z_mean)
#val_others_isof_scores = isof.score_samples(val_others_z_mean)




# Local Outlier Factor anomaly scores
# Local Outlier Factor anomaly scores
#in5_lof_scores = lof.negative_outlier_factor_(in5_z_log_var)
#in3_lof_scores = -lof.negative_outlier_factor_(in3_z_log_var)
#outlet_lof_scores = -lof.negative_outlier_factor_(outlet_z_log_var)
#val_outlet_lof_scores = -lof.negative_outlier_factor_(val_outlet_z_log_var)
#val_others_lof_scores = -lof.negative_outlier_factor_(val_others_z_log_var)

# Lower scores indicate more anomalous instances

In [ ]:
import seaborn as sns
plt.figure(figsize=(8, 6))

# Create the box plot
sns.boxplot(data=[in3_svm_scores, in5_svm_scores, outlet_svm_scores, val_outlet_svm_scores, val_others_svm_scores])

# Setting labels
#plt.xlabel('Class')
#plt.ylabel('cosine_similarity Scores')
plt.title('One-Class SVM scores')
plt.xticks([0, 1, 2, 3, 4], ['Inlet type-1', 'Inlet Type-2', 'Outlet', 'Outlet (Test)',  'Unmarked'])  # Label x-ticks with class names

# Show the plot
plt.show()

In [ ]:
import seaborn as sns
plt.figure(figsize=(8, 6))

# Create the box plot
sns.boxplot(data=[in3_isof_scores, in5_isof_scores, outlet_isof_scores, val_outlet_isof_scores,val_others_isof_scores])

# Setting labels
#plt.xlabel('Class')
#plt.ylabel('cosine_similarity Scores')
plt.title('Isolation forest scores')
plt.xticks([0, 1, 2, 3, 4], ['Inlet type-1', 'Inlet Type-2', 'Outlet', 'Outlet (Test)',  'Unmarked'])  # Label x-ticks with class names

# Show the plot
plt.show()

In [ ]:

errors = np.concatenate([val_outlet_mahalanobis_distances,val_in3_mahalanobis_distances,val_in5_mahalanobis_distances,val_others_mahalanobis_distances])

labels = np.concatenate([np.zeros(len(val_outlet_mahalanobis_distances)), np.ones(len(val_in3_mahalanobis_distances)+len(val_in5_mahalanobis_distances)+len(val_others_mahalanobis_distances))])

sv_errors = np.concatenate([val_outlet_svm_scores,val_in3_svm_scores,val_in5_svm_scores,val_others_svm_scores])
sv_errors = sv_errors*-1

elbo_errors= np.concatenate([val_outlet_elbo_losses,val_in3_elbo_losses,val_in5_elbo_losses,val_others_elbo_losses])

gmm_errors = np.concatenate([val_gmm_outlet,val_gmm_in3,val_gmmin5,val_gmm_others])*-1

kd_errors = np.concatenate([val_kd_outlet,val_kd_in3,val_kd_in5,val_kd_others])*-1

In [ ]:
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support

auc_roc = roc_auc_score(labels, errors)
sv_auc_roc = roc_auc_score(labels, sv_errors)
el_auc_roc=  roc_auc_score(labels, elbo_errors)
gmm_auc_roc=  roc_auc_score(labels, gmm_errors)
kd_auc_roc=  roc_auc_score(labels, kd_errors)
print(f"AUC-ROC: {auc_roc}")


In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, precision_recall_curve, f1_score, auc
from scipy.optimize import brentq
from scipy.interpolate import interp1d

sv_fpr,sv_tpr, sv_thresolds = roc_curve(labels, sv_errors) 
sv_eer = brentq(lambda x: 1. - x - interp1d(sv_fpr, sv_tpr)(x), 0., 1.)
sv_eer_threshold = interp1d(sv_fpr, sv_thresolds)(sv_eer)


el_fpr,el_tpr, el_thresolds = roc_curve(labels, elbo_errors) 
el_eer = brentq(lambda x: 1. - x - interp1d(el_fpr, el_tpr)(x), 0., 1.)
el_eer_threshold = interp1d(el_fpr, el_thresolds)(el_eer)


kd_fpr,kd_tpr, kd_thresolds = roc_curve(labels, kd_errors) 
kd_eer = brentq(lambda x: 1. - x - interp1d(kd_fpr, kd_tpr)(x), 0., 1.)
kd_eer_threshold = interp1d(kd_fpr, kd_thresolds)(kd_eer)

gmm_fpr,gmm_tpr, gmm_thresolds = roc_curve(labels, gmm_errors) 
gmm_eer = brentq(lambda x: 1. - x - interp1d(gmm_fpr, gmm_tpr)(x), 0., 1.)
gmm_eer_threshold = interp1d(gmm_fpr, gmm_thresolds)(gmm_eer)


fpr, tpr, thresholds = roc_curve(labels, errors)
eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
eer_threshold = interp1d(fpr, thresholds)(eer)



precision, recall, thresholds_pr = precision_recall_curve(labels, errors)
sv_precission, sv_recall, sv_thresholds_pr = precision_recall_curve(labels,sv_errors)
el_precission, el_recall, el_thresholds_pr = precision_recall_curve(labels,elbo_errors)
gmm_precission, gmm_recall, gmm_thresholds_pr = precision_recall_curve(labels,gmm_errors)
kd_precission, kd_recall, kd_thresholds_pr = precision_recall_curve(labels,kd_errors)



fscores = 2 * (precision * recall) / (precision + recall)
idx = np.nanargmax(fscores)  # handle NaNs if any
best_f1_score = fscores[idx]
f1_threshold = thresholds_pr[idx]


sv_fscores = 2 * (sv_precission * sv_recall) / (sv_precission + sv_recall)
sv_idx = np.nanargmax(sv_fscores)  # handle NaNs if any
sv_best_f1_score = sv_fscores[sv_idx]
sv_f1_threshold = sv_thresholds_pr[sv_idx]


el_fscores = 2 * (el_precission * el_recall) / (el_precission + el_recall)
el_idx = np.nanargmax(el_fscores)  # handle NaNs if any
el_best_f1_score = el_fscores[el_idx]
el_f1_threshold = el_thresholds_pr[el_idx]


gmm_fscores = 2 * (gmm_precission * gmm_recall) / (gmm_precission + gmm_recall)
gmm_idx = np.nanargmax(gmm_fscores)  # handle NaNs if any
gmm_best_f1_score = gmm_fscores[gmm_idx]
gmm_f1_threshold = el_thresholds_pr[gmm_idx]

kd_fscores = 2 * (kd_precission * kd_recall) / (kd_precission + kd_recall)
kd_idx = np.nanargmax(kd_fscores)  # handle NaNs if any
kd_best_f1_score = kd_fscores[kd_idx]
kd_f1_threshold = el_thresholds_pr[kd_idx]



best_precision_idx = np.argmax(precision)
best_recall_idx = np.argmax(recall)
precision_threshold = thresholds_pr[best_precision_idx-1]  # adjusting index as it's one less
recall_threshold = thresholds_pr[best_recall_idx-1]       # adjusting index as it's one less


#print(f"Equal Error Rate (EER): {eer:.5f}, Threshold: {eer_threshold}")
print(f"Best F1 Score elbo: {el_best_f1_score:.5f}")#, F1 Threshold: {f1_threshold}")
print(f"Best F1 Score maha: {best_f1_score:.5f}")#, F1 Threshold: {f1_threshold}")

print(f"Best F1 Score svm: {sv_best_f1_score:.5f}")#, F1 Threshold: {f1_threshold}")

print(f"Best F1 Score gmm: {gmm_best_f1_score:.5f}")#, F1 Threshold: {f1_threshold}")
print(f"Best F1 Score kd: {kd_best_f1_score:.5f}")#, F1 Threshold: {f1_threshold}")

#print(f"Best Precision: {precision[best_precision_idx]:.5f}")#, Precision Threshold: {precision_threshold}")
#print(f"Best Recall: {recall[best_recall_idx]:.5f}, Recall Threshold: {recall_threshold}")



In [ ]:
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt

plt.figure()


plt.plot(el_fpr, el_tpr, label=f'ELBO (area = {el_auc_roc:.7f})')
plt.plot(fpr, tpr, label=f'Mahalanobis distance (area = {auc_roc:.7f})')
plt.plot(sv_fpr,sv_tpr,label=f'One-class SVM (area = {sv_auc_roc:.7f})')
plt.plot(gmm_fpr, gmm_tpr, label=f'GMM (area = {gmm_auc_roc:.7f})')
plt.plot(kd_fpr, kd_tpr, label=f'KD (area = {kd_auc_roc:.7f})')

plt.plot()
plt.plot([0, 1], [0, 1], 'k--')
#plt.plot([eer], [1-eer], 'ro', label=f'EER = {eer:.5f}')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()


In [ ]:
print(el_eer)
print(eer)
print(sv_eer)
print(gmm_eer)
print(kd_eer)

In [ ]:
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve



eer_precision = interp1d(recall, precision)(eer)

auc_prc = auc(recall,precision)
sv_auc_prc = auc(sv_recall,sv_precission)
el_auc_prc = auc(el_recall,el_precission)
gmm_auc_prc = auc(gmm_recall,gmm_precission)
kd_auc_prc = auc(kd_recall,kd_precission)


plt.figure()
plt.plot(el_recall, el_precission, label=f'Elbo (area = {el_auc_prc:.7f})')
plt.plot(recall, precision, label=f'Mahalanobis distance (area = {auc_prc:.7f})')
plt.plot(sv_recall,sv_precission, label = f'One-class SVM (area = {sv_auc_prc:.7f})')
plt.plot(gmm_recall,gmm_precission, label = f'GMM (area = {gmm_auc_prc:.7f})')
plt.plot(kd_recall,kd_precission, label = f'KD (area = {kd_auc_prc:.7f})')
#plt.plot([eer], [eer_precision], 'ro', label=f'EER = {eer:.5f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="lower left")
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

# Assuming you've already computed 'eer_val' and 'thresh' as shown previously
predicted_labels = (errors>= eer_threshold).astype(int)

precision, recall, f1_score, _ = precision_recall_fscore_support(labels, predicted_labels, average='binary')
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1_score}")


In [ ]:
# Assuming you've already computed 'eer_val' and 'thresh' as shown previously
predicted_labels = (errors>= min(val_others_mahalanobis_distances)-0.0000001).astype(int)

precision, recall, f1_score, _ = precision_recall_fscore_support(labels, predicted_labels, average='binary')
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1_score}")

    Firstly, the KL divergence loss term keeps z_mean and z_log_var values close to a standard normal distribution. Secondly, the stochastic encoder leads to a more continuous latent space, reducing the occurrence of poorly formed images.


Finally, when we color-code points in the latent space by clothing type, we observe that there is no bias favoring any specific type. The right-hand plot displays the space converted into p-values, revealing nearly equal representation for each color. It's crucial to emphasize that the model learned clothing distinctions independently, without using labels during training, to minimize reconstruction loss.